In [1]:
# Celda 1: Importar librerías
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectFromModel
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

✅ Librerías importadas


In [2]:
# Celda 2: Clase del clasificador mejorado para tráfico real
class RealTrafficClassifier:
    """Clasificador optimizado para tráfico HTTP/HTTPS real"""
    
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.target_encoder = LabelEncoder()
        self.feature_selector = None
        self.is_trained = False
        
    def create_synthetic_training_data(self):
        """Crear datos de entrenamiento sintéticos basados en patrones reales"""
        print("🎯 Creando dataset de entrenamiento sintético...")
        
        # Patrones de tráfico normal
        normal_patterns = [
            {'method': 'GET', 'path': '/', 'status': 200, 'size': 1500},
            {'method': 'GET', 'path': '/index.html', 'status': 200, 'size': 2000},
            {'method': 'GET', 'path': '/images/logo.png', 'status': 200, 'size': 5000},
            {'method': 'POST', 'path': '/login', 'status': 200, 'size': 800},
            {'method': 'GET', 'path': '/products', 'status': 200, 'size': 3000}
        ]
        
        # Patrones de ataques
        attack_patterns = {
            'sql_injection': [
                {'method': 'GET', 'path': "/search?q=' OR '1'='1", 'status': 200, 'size': 500},
                {'method': 'GET', 'path': "/admin'--", 'status': 200, 'size': 600},
                {'method': 'POST', 'path': '/login', 'status': 401, 'size': 400},
                {'method': 'GET', 'path': "/union select 1,2,3--", 'status': 200, 'size': 700}
            ],
            'xss': [
                {'method': 'GET', 'path': '/search?q=<script>alert(1)</script>', 'status': 200, 'size': 800},
                {'method': 'POST', 'path': '/comment', 'status': 200, 'size': 900},
                {'method': 'GET', 'path': '/?param=javascript:alert(1)', 'status': 200, 'size': 600}
            ],
            'path_traversal': [
                {'method': 'GET', 'path': '/../../etc/passwd', 'status': 404, 'size': 300},
                {'method': 'GET', 'path': '/..%2f..%2fwin.ini', 'status': 404, 'size': 350},
                {'method': 'GET', 'path': '/../../../config.php', 'status': 404, 'size': 400}
            ],
            'dos': [
                {'method': 'GET', 'path': '/', 'status': 200, 'size': 100},
                {'method': 'POST', 'path': '/api', 'status': 200, 'size': 50},
                {'method': 'GET', 'path': '/images/1.jpg', 'status': 200, 'size': 80}
            ]
        }
        
        data = []
        labels = []
        
        # Generar datos normales
        for _ in range(1000):
            pattern = normal_patterns[np.random.randint(0, len(normal_patterns))]
            features = self._extract_features_from_pattern(pattern, 'normal')
            data.append(features)
            labels.append('normal')
        
        # Generar datos de ataques
        for attack_type, patterns in attack_patterns.items():
            for _ in range(250):
                pattern = patterns[np.random.randint(0, len(patterns))]
                features = self._extract_features_from_pattern(pattern, attack_type)
                data.append(features)
                labels.append(attack_type)
        
        df = pd.DataFrame(data)
        return df, pd.Series(labels)
    
    def _extract_features_from_pattern(self, pattern, label):
        """Extraer características de un patrón de tráfico"""
        method = pattern['method']
        path = pattern['path']
        status = pattern['status']
        size = pattern['size']
        
        # Características basadas en el patrón
        features = {
            'method_encoded': self._encode_method(method),
            'path_length': len(path),
            'has_special_chars': int(any(c in path for c in ["'", '"', '<', '>', '(', ')', ';'])),
            'has_encoding': int('%' in path),
            'has_sql_keywords': int(any(kw in path.lower() for kw in ['select', 'union', 'insert', 'drop', 'or '])),
            'has_script_tags': int('<script' in path.lower()),
            'has_path_traversal': int(any(pt in path for pt in ['../', '..\\', 'etc/passwd'])),
            'status_code': status,
            'response_size': size,
            'is_error': int(status >= 400),
            'request_frequency': np.random.poisson(5),  # Frecuencia de requests
            'unique_paths_ratio': np.random.uniform(0.1, 1.0),
            'error_rate': np.random.uniform(0.0, 0.3) if label == 'normal' else np.random.uniform(0.1, 0.8)
        }
        
        return features
    
    def _encode_method(self, method):
        """Codificar método HTTP"""
        method_map = {'GET': 0, 'POST': 1, 'PUT': 2, 'DELETE': 3, 'HEAD': 4}
        return method_map.get(method, 5)
    
    def train_flexible_model(self):
        """Entrenar modelo con datos sintéticos y reales"""
        print("🤖 Entrenando modelo flexible...")
        
        # Crear datos de entrenamiento
        X_train, y_train = self.create_synthetic_training_data()
        
        # Procesamiento
        X_processed = self._preprocess_features(X_train)
        y_encoded = self.target_encoder.fit_transform(y_train)
        
        # Selección de características
        if X_processed.shape[1] > 10:
            print("🔍 Realizando selección de características...")
            self.feature_selector = SelectFromModel(
                RandomForestClassifier(n_estimators=50, random_state=42),
                threshold='median'
            )
            X_processed = self.feature_selector.fit_transform(X_processed, y_encoded)
            print(f"   Features seleccionados: {X_processed.shape[1]}")
        
        # Modelo
        self.model = RandomForestClassifier(
            n_estimators=150,
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        
        # Validación cruzada
        cv_scores = cross_val_score(self.model, X_processed, y_encoded, cv=5, scoring='accuracy')
        print(f"📊 Validación Cruzada: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
        
        # Entrenamiento final
        self.model.fit(X_processed, y_encoded)
        self.is_trained = True
        
        print(f"✅ Modelo entrenado - {len(self.target_encoder.classes_)} clases")
        print(f"📋 Clases: {list(self.target_encoder.classes_)}")
        return self
    
    def _preprocess_features(self, X):
        """Preprocesar características"""
        # Escalar características numéricas
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            X_scaled = self.scaler.fit_transform(X)
            return X_scaled
        return X.values
    
    def extract_features_from_http(self, request_data):
        """Extraer características de tráfico HTTP real"""
        try:
            method = request_data.get('method', 'GET')
            url = request_data.get('url', '')
            path = request_data.get('path', '')
            status_code = request_data.get('status_code', 200)
            content_length = request_data.get('content_length', 0)
            query_params = request_data.get('query_params', {})
            
            # Análisis de URL y path
            full_path = (url + path).lower()
            
            # Detección de patrones de ataque
            sql_patterns = [r"'.*?(union|select|insert|update|delete|drop|exec).*?'",
                          r"'.*?(\-\-|#|\/\*).*?'", r"'.*?(1=1|2=2|0=0).*?'"]
            xss_patterns = [r"<script.*?>.*?</script>", r"javascript:", r"onload=.*?", r"alert\(.*?\)"]
            traversal_patterns = [r"\.\.\/\.\.\/", r"\.\.\\\.\.\\", r"etc/passwd", r"win\.ini"]
            
            features = {
                'method_encoded': self._encode_method(method),
                'path_length': len(path),
                'url_length': len(url),
                'has_special_chars': int(any(c in full_path for c in ["'", '"', '<', '>', '(', ')', ';'])),
                'has_encoding': int('%' in full_path),
                'has_sql_keywords': int(any(re.search(pattern, full_path, re.IGNORECASE) for pattern in sql_patterns)),
                'has_script_tags': int(any(re.search(pattern, full_path, re.IGNORECASE) for pattern in xss_patterns)),
                'has_path_traversal': int(any(re.search(pattern, full_path, re.IGNORECASE) for pattern in traversal_patterns)),
                'status_code': status_code,
                'response_size': content_length,
                'is_error': int(status_code >= 400),
                'num_parameters': len(query_params),
                'has_suspicious_params': int(any('cmd' in k.lower() or 'exec' in k.lower() or 'union' in k.lower() 
                                               for k in query_params.keys())),
                'request_complexity': len(path.split('/')) + len(query_params)
            }
            
            return features
            
        except Exception as e:
            print(f"❌ Error extrayendo características: {e}")
            return {}
    
    def predict_http_traffic(self, request_data):
        """Predecir tipo de ataque para tráfico HTTP"""
        if not self.is_trained:
            return self._rule_based_detection(request_data)
        
        try:
            # Extraer características
            features = self.extract_features_from_http(request_data)
            if not features:
                return self._rule_based_detection(request_data)
            
            # Convertir a DataFrame
            feature_df = pd.DataFrame([features])
            
            # Preprocesar
            X_processed = self._preprocess_features(feature_df)
            
            # Aplicar selección de características si existe
            if self.feature_selector is not None:
                X_processed = self.feature_selector.transform(X_processed)
            
            # Predecir
            prediction_encoded = self.model.predict(X_processed)[0]
            probability = float(max(self.model.predict_proba(X_processed)[0]))
            prediction = self.target_encoder.inverse_transform([prediction_encoded])[0]
            
            # Mapear a categorías de ataque
            attack_mapping = {
                'normal': {'prediction': 0, 'type': 'Normal'},
                'sql_injection': {'prediction': 1, 'type': 'SQL Injection'},
                'xss': {'prediction': 2, 'type': 'XSS'},
                'path_traversal': {'prediction': 3, 'type': 'Path Traversal'},
                'dos': {'prediction': 4, 'type': 'DoS'}
            }
            
            result = attack_mapping.get(prediction, {'prediction': 0, 'type': 'Normal'})
            result['confidence'] = probability
            result['real_ml'] = True
            result['success'] = True
            
            return result
            
        except Exception as e:
            print(f"❌ Error en predicción ML: {e}")
            return self._rule_based_detection(request_data)
    
    def _rule_based_detection(self, request_data):
        """Detección basada en reglas como fallback"""
        features = self.extract_features_from_http(request_data)
        
        if features.get('has_sql_keywords', 0):
            return {"prediction": 1, "confidence": 0.85, "attack_type": "SQL Injection", "real_ml": False, "success": True}
        elif features.get('has_script_tags', 0):
            return {"prediction": 2, "confidence": 0.80, "attack_type": "XSS", "real_ml": False, "success": True}
        elif features.get('has_path_traversal', 0):
            return {"prediction": 3, "confidence": 0.75, "attack_type": "Path Traversal", "real_ml": False, "success": True}
        elif features.get('is_error', 0) and features.get('request_complexity', 0) > 10:
            return {"prediction": 4, "confidence": 0.70, "attack_type": "DoS", "real_ml": False, "success": True}
        else:
            return {"prediction": 0, "confidence": 0.90, "attack_type": "Normal", "real_ml": False, "success": True}
    
    def save_model(self, filepath):
        """Guardar modelo entrenado"""
        if not self.is_trained:
            raise ValueError("No hay modelo entrenado para guardar.")
            
        model_data = {
            'model': self.model,
            'scaler': self.scaler,
            'target_encoder': self.target_encoder,
            'feature_selector': self.feature_selector,
            'is_trained': self.is_trained
        }
        
        joblib.dump(model_data, filepath)
        print(f"💾 Modelo guardado en: {filepath}")
    
    def load_model(self, filepath):
        """Cargar modelo entrenado"""
        model_data = joblib.load(filepath)
        
        self.model = model_data['model']
        self.scaler = model_data['scaler']
        self.target_encoder = model_data['target_encoder']
        self.feature_selector = model_data['feature_selector']
        self.is_trained = model_data['is_trained']
        
        print(f"📂 Modelo cargado - {len(self.target_encoder.classes_)} clases")
        return self

print("✅ Clasificador para tráfico real definido")

✅ Clasificador para tráfico real definido


In [3]:
# Celda 3: Entrenar y evaluar el modelo
# Entrenar modelo
classifier = RealTrafficClassifier()
classifier.train_flexible_model()

# Probar con datos de ejemplo
test_request = {
    'method': 'GET',
    'url': 'http://testphp.vulnweb.com',
    'path': "/search?q=' OR '1'='1",
    'status_code': 200,
    'content_length': 500,
    'query_params': {'q': "' OR '1'='1"}
}

prediction = classifier.predict_http_traffic(test_request)
print(f"🎯 Predicción de ejemplo: {prediction}")

# Guardar modelo
classifier.save_model('../backend/real_traffic_model.pkl')

🤖 Entrenando modelo flexible...
🎯 Creando dataset de entrenamiento sintético...
🔍 Realizando selección de características...
   Features seleccionados: 7
📊 Validación Cruzada: 1.000 (±0.000)
✅ Modelo entrenado - 5 clases
📋 Clases: ['dos', 'normal', 'path_traversal', 'sql_injection', 'xss']
❌ Error extrayendo características: name 're' is not defined
❌ Error extrayendo características: name 're' is not defined
🎯 Predicción de ejemplo: {'prediction': 0, 'confidence': 0.9, 'attack_type': 'Normal', 'real_ml': False, 'success': True}
💾 Modelo guardado en: ../backend/real_traffic_model.pkl
